# INFORMACIÓN DEL PROYECTO

### TÍTULO

In [1]:
#Escribe el título del proyecto en Markdown
titulo = "Sistema Híbrido de Machine Learning para la Clasificación y Reasignación de Defectos Superficiales en Placas de Acero"

### TEMA

In [2]:
# Describe brevemente el tema del proyecto en Markdown
tema = """
Implementación de un pipeline de Machine Learning (aprendizaje supervisado y no supervisado) 
aplicado al control de calidad en siderurgia. El proyecto da continuidad al EDA previo, 
buscando resolver la 'ceguera operativa' del sistema de visión artificial actual (que agrupa 
el 34.7% de los fallos totales y el 72.4% en placas gruesas bajo la etiqueta genérica 'Otro') 
mediante modelos predictivos capaces de clasificar defectos conocidos y descubrir nuevas 
tipologías de fallos ocultos.
"""


### OBJETIVOS

Define aquí el objetivo del proyecto

### OBJETIVO PRINCIPAL
Desarrollar un modelo de Machine Learning para clasificar automáticamente los defectos en placas de acero y reducir los fallos sin identificar (`Otro`), generando un ahorro de hasta **6.930 € por tonelada** (Regla 10X).

### OBJETIVOS ESPECÍFICOS
1. **Clasificación de defectos conocidos:** Entrenar y optimizar modelos supervisados usando solo los 6 defectos reales etiquetados (65.3% de los datos), corrigiendo el desbalanceo de clases y las variables redundantes.
2. **Modelos por espesor vs. modelo único:** Comprobar si entrenar 3 modelos separados según el grosor de la placa ($\le 60\text{ mm}$, $61\text{-}150\text{ mm}$, $> 150\text{ mm}$) da mejores resultados que usar un solo modelo general.
3. **Rescate de la categoría `Otro`:** Usar las probabilidades del mejor modelo para reasignar los registros marcados como `Otro` (34.7% del total) que sí correspondan a defectos conocidos.
4. **Búsqueda de nuevos fallos (Clustering):** Agrupar los registros de `Otro` que sigan sin clasificarse (sobre todo en placas gruesas $> 150\text{ mm}$) para descubrir posibles tipos de defectos nuevos.



## OBTENCIÓN DE LOS DATOS

### DATASETS Y FUENTES ALTERNATIVAS DE DATOS

Incluye aquí una vista del dataset o datasets de los que partirás para poder evaluar tu hipótesis. <br>
También incluye el origen de estos datos y su fuente.

In [7]:
"""
- Dataset y Fuente: 'Steel Plates Faults' (steel-faults.csv), obtenido de Kaggle / UCI (recolectado por Semeion Research Center mediante cámaras de visión artificial).
- Estructura (1.941 registros y 34 variables sin nulos):
  * 27 variables predictoras: Coordenadas (X, Y), tamaño/píxeles, luminosidad, morfología y parámetros del material (Espesor, Tipo_Acero).
  * 7 clases objetivo: Pliegue, Raya, Surco, Mancha, Suciedad, Abolladura y Otro.
- División para Machine Learning:
  * df_conocidos (1.267 registros): Los 6 defectos reales para entrenar y evaluar los modelos.
  * df_otros (674 registros): Registros etiquetados como 'Otro' para reclasificarlos o agruparlos mediante clustering.
"""

"\n- Dataset y Fuente: 'Steel Plates Faults' (steel-faults.csv), obtenido de Kaggle / UCI (recolectado por Semeion Research Center mediante cámaras de visión artificial).\n- Estructura (1.941 registros y 34 variables sin nulos):\n  * 27 variables predictoras: Coordenadas (X, Y), tamaño/píxeles, luminosidad, morfología y parámetros del material (Espesor, Tipo_Acero).\n  * 7 clases objetivo: Pliegue, Raya, Surco, Mancha, Suciedad, Abolladura y Otro.\n- División para Machine Learning:\n  * df_conocidos (1.267 registros): Los 6 defectos reales para entrenar y evaluar los modelos.\n  * df_otros (674 registros): Registros etiquetados como 'Otro' para reclasificarlos o agruparlos mediante clustering.\n"

Muestra mediante un head() los principales datasets con los que vas a trabajar

In [10]:
import pandas as pd

#Dataset original
df_raw = pd.read_csv('data/steel-faults.csv') # Ajusta la ruta si estás dentro de /notebooks/ -> '../data/steel-faults.csv'

#Vble objetivo (misma lógica del EDA)
defect_cols = ['Pastry', 'Z_Scratch', 'K_Scatch', 'Stains', 'Dirtiness', 'Bumps', 'Other_Faults']
df_raw['Tipo_Defecto'] = df_raw[defect_cols].idxmax(axis=1)

#División estratégica para el proyecto de Machine Learning
df_conocidos = df_raw[df_raw['Tipo_Defecto'] != 'Other_Faults'].copy()
df_otros = df_raw[df_raw['Tipo_Defecto'] == 'Other_Faults'].copy()

print(f"Total registros: {len(df_raw)} | Conocidos: {len(df_conocidos)} | Otros: {len(df_otros)}")


Total registros: 1941 | Conocidos: 1268 | Otros: 673


In [11]:
df_raw.head()

,X_Minimum,X_Maximum,Y_Minimum,Y_Maximum,Pixels_Areas,X_Perimeter,Y_Perimeter,Sum_of_Luminosity,Minimum_of_Luminosity,Maximum_of_Luminosity,...,Luminosity_Index,SigmoidOfAreas,Pastry,Z_Scratch,K_Scatch,Stains,Dirtiness,Bumps,Other_Faults,Tipo_Defecto
0,42,50,270900,270944,267,17,44,24220,76,108,...,-0.2913,0.5822,1,0,0,0,0,0,0,Pastry
1,645,651,2538079,2538108,108,10,30,11397,84,123,...,-0.1756,0.2984,1,0,0,0,0,0,0,Pastry
2,829,835,1553913,1553931,71,8,19,7972,99,125,...,-0.1228,0.2150,1,0,0,0,0,0,0,Pastry
3,853,860,369370,369415,176,13,45,18996,99,126,...,-0.1568,0.5212,1,0,0,0,0,0,0,Pastry
4,1289,1306,498078,498335,2409,60,260,246930,37,126,...,-0.1992,1.0000,1,0,0,0,0,0,0,Pastry


In [12]:
df_conocidos.head()

,X_Minimum,X_Maximum,Y_Minimum,Y_Maximum,Pixels_Areas,X_Perimeter,Y_Perimeter,Sum_of_Luminosity,Minimum_of_Luminosity,Maximum_of_Luminosity,...,Luminosity_Index,SigmoidOfAreas,Pastry,Z_Scratch,K_Scatch,Stains,Dirtiness,Bumps,Other_Faults,Tipo_Defecto
0,42,50,270900,270944,267,17,44,24220,76,108,...,-0.2913,0.5822,1,0,0,0,0,0,0,Pastry
1,645,651,2538079,2538108,108,10,30,11397,84,123,...,-0.1756,0.2984,1,0,0,0,0,0,0,Pastry
2,829,835,1553913,1553931,71,8,19,7972,99,125,...,-0.1228,0.2150,1,0,0,0,0,0,0,Pastry
3,853,860,369370,369415,176,13,45,18996,99,126,...,-0.1568,0.5212,1,0,0,0,0,0,0,Pastry
4,1289,1306,498078,498335,2409,60,260,246930,37,126,...,-0.1992,1.0000,1,0,0,0,0,0,0,Pastry


In [13]:
df_otros.head()

,X_Minimum,X_Maximum,Y_Minimum,Y_Maximum,Pixels_Areas,X_Perimeter,Y_Perimeter,Sum_of_Luminosity,Minimum_of_Luminosity,Maximum_of_Luminosity,...,Luminosity_Index,SigmoidOfAreas,Pastry,Z_Scratch,K_Scatch,Stains,Dirtiness,Bumps,Other_Faults,Tipo_Defecto
1268,1108,1120,2497122,2497138,137,13,16,15672,85,133,...,-0.1063,0.3241,0,0,0,0,0,0,1,Other_Faults
1269,333,342,764107,764213,525,16,106,57020,95,127,...,-0.1515,0.9872,0,0,0,0,0,0,1,Other_Faults
1270,1269,1296,374730,374925,2463,188,212,296673,42,143,...,-0.0590,1.0000,0,0,0,0,0,0,1,Other_Faults
1271,1109,1144,401768,401951,4298,188,189,515942,6,148,...,-0.0622,1.0000,0,0,0,0,0,0,1,Other_Faults
1272,1293,1316,600561,600841,2643,170,285,271889,44,125,...,-0.1963,1.0000,0,0,0,0,0,0,1,Other_Faults
